In [22]:
import kagglehub
import os

tikharm_32_path = kagglehub.dataset_download(
    "aryansraut/tikharm-data-32-frames"
)

print("Dataset path:")
print(tikharm_32_path)

print("\nContents:")
for item in os.listdir(tikharm_32_path):
    print(item)

Dataset path:
/kaggle/input/datasets/aryansraut/tikharm-data-32-frames

Contents:
TikHarm_frames_32
TikHarm_std
tikharm_metadata.csv
TikHarm_audio


In [34]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchvision.transforms as T

from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

BASE = "/kaggle/input/datasets/aryansraut/tikharm-data-32-frames"

VISUAL_ROOT = f"{BASE}/TikHarm_frames_32/TikHarm_frames_32"
AUDIO_ROOT = f"{BASE}/TikHarm_audio/TikHarm_audio"

VISUAL_CKPT = "/kaggle/input/models/arsalan9702/tikharm-models-new/pytorch/default/1/best_swin3d_tikharm.pt"
AUDIO_CKPT  = "/kaggle/input/models/arsalan9702/tikharm-models-new/pytorch/default/1/best_audio_cnn14.pth"

NUM_FRAMES = 32
NUM_CLASSES = 4
MAX_AUDIO = 16000 * 10
BATCH_SIZE = 8

CLASSES = [
    "Adult Content",
    "Harmful Content",
    "Safe",
    "Suicide"
]

CLASS_TO_IDX = {
    cls: i for i, cls in enumerate(CLASSES)
}

print("Visual root:", VISUAL_ROOT)
print("Audio root:", AUDIO_ROOT)



Device: cuda
GPU: Tesla T4
Visual root: /kaggle/input/datasets/aryansraut/tikharm-data-32-frames/TikHarm_frames_32/TikHarm_frames_32
Audio root: /kaggle/input/datasets/aryansraut/tikharm-data-32-frames/TikHarm_audio/TikHarm_audio


In [35]:
!pip install torchlibrosa -q

In [36]:
from torchlibrosa.stft import Spectrogram, LogmelFilterBank


class CNN14(nn.Module):
    def __init__(self, classes_num=4):
        super().__init__()

        self.spectrogram_extractor = Spectrogram(
            n_fft=1024,
            hop_length=320,
            win_length=1024,
            window="hann",
            center=True,
            pad_mode="reflect"
        )

        self.logmel_extractor = LogmelFilterBank(
            sr=16000,
            n_fft=1024,
            n_mels=64,
            fmin=50,
            fmax=8000
        )

        self.bn0 = nn.BatchNorm2d(64)

        self.conv_block1 = self._conv_block(1, 64)
        self.conv_block2 = self._conv_block(64, 128)
        self.conv_block3 = self._conv_block(128, 256)
        self.conv_block4 = self._conv_block(256, 512)

        self.fc1 = nn.Linear(512, 512)
        self.fc_out = nn.Linear(512, classes_num)

    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)
        )

    def forward(self, x):
        x = self.spectrogram_extractor(x)
        x = self.logmel_extractor(x)

        x = x.transpose(1, 3)
        x = self.bn0(x)
        x = x.transpose(1, 3)

        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)

        x = torch.mean(x, dim=3)
        x = torch.mean(x, dim=2)

        x = F.relu(self.fc1(x))

        return self.fc_out(x)

In [37]:
class FusionDataset(Dataset):
    def __init__(
        self,
        visual_root,
        audio_root,
        split,
        num_frames=32,
        max_audio=16000 * 10
    ):
        self.samples = []
        self.num_frames = num_frames
        self.max_audio = max_audio

        self.transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        for cls in CLASSES:
            visual_class_dir = os.path.join(
                visual_root, split, cls
            )

            audio_class_dir = os.path.join(
                audio_root, split, cls
            )

            if not os.path.isdir(visual_class_dir):
                raise FileNotFoundError(
                    f"Missing directory: {visual_class_dir}"
                )

            for video_id in sorted(os.listdir(visual_class_dir)):
                video_dir = os.path.join(
                    visual_class_dir, video_id
                )

                if not os.path.isdir(video_dir):
                    continue

                audio_path = os.path.join(
                    audio_class_dir,
                    video_id + ".wav"
                )

                if os.path.isfile(audio_path):
                    self.samples.append(
                        (
                            video_dir,
                            audio_path,
                            CLASS_TO_IDX[cls]
                        )
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        video_dir, audio_path, label = self.samples[index]

        frame_files = sorted(
            f for f in os.listdir(video_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )

        if len(frame_files) == 0:
            raise RuntimeError(f"No frames found: {video_dir}")

        if len(frame_files) >= self.num_frames:
            indices = torch.linspace(
                0,
                len(frame_files) - 1,
                self.num_frames
            ).long()

            selected_frames = [
                frame_files[i] for i in indices
            ]
        else:
            selected_frames = (
                frame_files +
                [frame_files[-1]] *
                (self.num_frames - len(frame_files))
            )

        frames = []

        for frame_file in selected_frames:
            image = Image.open(
                os.path.join(video_dir, frame_file)
            ).convert("RGB")

            frames.append(self.transform(image))

        video = torch.stack(frames)
        video = video.permute(1, 0, 2, 3)

        waveform, sample_rate = torchaudio.load(audio_path)

        if sample_rate != 16000:
            waveform = torchaudio.functional.resample(
                waveform,
                sample_rate,
                16000
            )

        waveform = waveform.mean(dim=0)

        if waveform.shape[0] < self.max_audio:
            waveform = F.pad(
                waveform,
                (0, self.max_audio - waveform.shape[0])
            )
        else:
            waveform = waveform[:self.max_audio]

        return video, waveform, label

In [38]:
train_dataset = FusionDataset(
    VISUAL_ROOT,
    AUDIO_ROOT,
    "train",
    NUM_FRAMES,
    MAX_AUDIO
)

val_dataset = FusionDataset(
    VISUAL_ROOT,
    AUDIO_ROOT,
    "val",
    NUM_FRAMES,
    MAX_AUDIO
)

test_dataset = FusionDataset(
    VISUAL_ROOT,
    AUDIO_ROOT,
    "test",
    NUM_FRAMES,
    MAX_AUDIO
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

Train: 2761
Val: 396
Test: 790


In [39]:
video, audio, label = train_dataset[0]

print("Video shape:", video.shape)
print("Audio shape:", audio.shape)
print("Label:", label)

assert video.shape == (3, 32, 224, 224)
assert audio.shape[0] == MAX_AUDIO

print("32-frame dataset verified.")

Video shape: torch.Size([3, 32, 224, 224])
Audio shape: torch.Size([160000])
Label: 0
32-frame dataset verified.


In [40]:
visual_model = torch.hub.load(
    "pytorch/vision:v0.15.2",
    "swin3d_t",
    pretrained=False
)

visual_model.head = nn.Linear(
    visual_model.head.in_features,
    NUM_CLASSES
)

visual_checkpoint = torch.load(
    VISUAL_CKPT,
    map_location=device
)

visual_model.load_state_dict(
    visual_checkpoint["model_state_dict"]
)

visual_model = visual_model.to(device)


audio_model = CNN14(NUM_CLASSES)

audio_checkpoint = torch.load(
    AUDIO_CKPT,
    map_location=device
)

audio_model.load_state_dict(audio_checkpoint)

audio_model = audio_model.to(device)


for p in visual_model.parameters():
    p.requires_grad = False

for p in audio_model.parameters():
    p.requires_grad = False

visual_model.eval()
audio_model.eval()

print("Both pretrained models loaded and frozen.")

Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.15.2


Both pretrained models loaded and frozen.


In [41]:
def get_visual_features(x):
    x = visual_model.patch_embed(x)
    x = visual_model.pos_drop(x)

    for layer in visual_model.features:
        x = layer(x)

    x = visual_model.norm(x)

    # Swin3D output: B, T, H, W, C
    x = x.mean(dim=(1, 2, 3))

    return x


def get_audio_features(x):
    x = audio_model.spectrogram_extractor(x)
    x = audio_model.logmel_extractor(x)

    x = x.transpose(1, 3)
    x = audio_model.bn0(x)
    x = x.transpose(1, 3)

    x = audio_model.conv_block1(x)
    x = audio_model.conv_block2(x)
    x = audio_model.conv_block3(x)
    x = audio_model.conv_block4(x)

    x = x.mean(dim=3)
    x = x.mean(dim=2)

    x = F.relu(audio_model.fc1(x))

    return x

In [42]:
with torch.no_grad():
    sample_video = video.unsqueeze(0).to(device)
    sample_audio = audio.unsqueeze(0).to(device)

    vf = get_visual_features(sample_video)
    af = get_audio_features(sample_audio)

print("Visual features:", vf.shape)
print("Audio features:", af.shape)

Visual features: torch.Size([1, 768])
Audio features: torch.Size([1, 512])


In [43]:
class FeatureFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(768 + 512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 4)
        )

    def forward(self, visual_features, audio_features):
        x = torch.cat(
            [visual_features, audio_features],
            dim=1
        )

        return self.network(x)


fusion_model = FeatureFusion().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    fusion_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [44]:
def train_one_epoch():
    fusion_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress = tqdm(
        train_loader,
        desc="Training"
    )

    for videos, audios, labels in progress:
        videos = videos.to(device, non_blocking=True)
        audios = audios.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.no_grad():
            vf = get_visual_features(videos)
            af = get_audio_features(audios)

        logits = fusion_model(vf, af)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / total, correct / total

In [45]:
def evaluate(loader):
    fusion_model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for videos, audios, labels in loader:
            videos = videos.to(device, non_blocking=True)
            audios = audios.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            vf = get_visual_features(videos)
            af = get_audio_features(audios)

            logits = fusion_model(vf, af)
            loss = criterion(logits, labels)

            running_loss += loss.item() * labels.size(0)

            predictions = logits.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

In [46]:
NUM_EPOCHS = 10
PATIENCE = 3

best_val_acc = 0.0
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = evaluate(val_loader)

    print()
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc:  {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Acc:    {val_acc:.4f}")
    print("-" * 40)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        torch.save(
            fusion_model.state_dict(),
            "best_feature_fusion_32frames.pth"
        )

        print("Best model saved.")

    else:
        patience_counter += 1
        print(
            f"No improvement: "
            f"{patience_counter}/{PATIENCE}"
        )

    if patience_counter >= PATIENCE:
        print("Early stopping.")
        break

Training: 100%|██████████| 346/346 [16:57<00:00,  2.94s/it, loss=0.0000]



Epoch 1/10
Train Loss: 0.0981
Train Acc:  0.9739
Val Loss:   0.6937
Val Acc:    0.8813
----------------------------------------
Best model saved.


Training: 100%|██████████| 346/346 [12:32<00:00,  2.17s/it, loss=0.0000]



Epoch 2/10
Train Loss: 0.0506
Train Acc:  0.9877
Val Loss:   0.6617
Val Acc:    0.8864
----------------------------------------
Best model saved.


Training: 100%|██████████| 346/346 [12:16<00:00,  2.13s/it, loss=0.0000]


...


Epoch 5/10
Train Loss: 0.0385
Train Acc:  0.9880
Val Loss:   0.7780
Val Acc:    0.8864
----------------------------------------
No improvement: 1/3


Training: 100%|██████████| 346/346 [12:21<00:00,  2.14s/it, loss=0.0000]



Epoch 6/10
Train Loss: 0.0255
Train Acc:  0.9942
Val Loss:   0.8144
Val Acc:    0.8838
----------------------------------------
No improvement: 2/3


Training: 100%|██████████| 346/346 [11:48<00:00,  2.05s/it, loss=0.0000]



Epoch 7/10
Train Loss: 0.0357
Train Acc:  0.9899
Val Loss:   1.0528
Val Acc:    0.8763
----------------------------------------
No improvement: 3/3
Early stopping.


In [ ]:
fusion_model.load_state_dict(
    torch.load(
        "best_feature_fusion_32frames.pth",
        map_location=device
    )
)

test_loss, test_acc = evaluate(test_loader)

print(f"Best Val Accuracy: {best_val_acc:.4f}")
print(f"Test Accuracy:     {test_acc:.4f}")

In [48]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score
)
import numpy as np


def classification_report_full(loader):
    fusion_model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for videos, audios, labels in loader:
            videos = videos.to(device)
            audios = audios.to(device)

            vf = get_visual_features(videos)
            af = get_audio_features(audios)

            logits = fusion_model(vf, af)
            predictions = logits.argmax(dim=1)

            y_true.extend(labels.numpy())
            y_pred.extend(predictions.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    print("=" * 60)
    print("FEATURE FUSION — 32 FRAMES")
    print("=" * 60)

    print(
        f"Accuracy: {accuracy_score(y_true, y_pred):.4f}"
    )

    print(
        f"Balanced Accuracy: "
        f"{balanced_accuracy_score(y_true, y_pred):.4f}"
    )

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=CLASSES,
            digits=4
        )
    )

    print("Confusion Matrix:")
    print(
        confusion_matrix(y_true, y_pred)
    )


classification_report_full(test_loader)

FEATURE FUSION — 32 FRAMES
Accuracy: 0.8671
Balanced Accuracy: 0.8671

Classification Report:
                 precision    recall  f1-score   support

  Adult Content     0.8543    0.8718    0.8629       195
Harmful Content     0.8929    0.7576    0.8197       198
           Safe     0.8626    0.9100    0.8856       200
        Suicide     0.8632    0.9289    0.8949       197

       accuracy                         0.8671       790
      macro avg     0.8682    0.8671    0.8658       790
   weighted avg     0.8683    0.8671    0.8658       790

Confusion Matrix:
[[170   8   9   8]
 [ 20 150  13  15]
 [  4   8 182   6]
 [  5   2   7 183]]
